# 语言模型预训练：从 Next-Token 目标到可恢复训练

> **本章定位**：以 Decoder-only Causal Language Model（因果语言模型）为主线，建立可评估、可调度、可恢复的预训练闭环。

> **两条互补路径**：第 2～4 节使用极小语料和原理模型拆解每个状态；第 5 节默认运行固定版本的 WikiText-2、GPT-2 Fast Tokenizer 与约 724 万参数的随机初始化模型，执行 1,500 次真实更新，而不是 8-step smoke test。

> **章节边界**：大规模数据工程、资源预算和训练系统优化分别见 `A20_data_engineering.ipynb`、`A10_resource_planning.ipynb` 与 `A40_training_optimization.ipynb`；SFT、DPO 与 GRPO 属于 `41_post_training.ipynb` 的后训练范围。

> **总览**：内容贯通版本化文本、Tokenizer 编码、定长切块、Decoder-only 前向传播、移位交叉熵、AdamW、Warmup + Cosine 调度、训练评估、完整状态 Checkpoint 与 Hugging Face `Trainer` 映射。

## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 模型训练与适配：预训练专题；训练系统前置 |
| 本章定位 | 把 Decoder-only 模型放入可评估、可调度、可恢复的预训练闭环。 |
| 先修知识 | 完成 `31`；理解训练循环、next-token loss 和数据划分。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 原理路径可在 CPU 运行；第 5 节真实训练在常见 Colab/Apple 加速器上的目标耗时约 1～3 分钟，CPU 通常更久，实际时间取决于硬件与 Checkpoint I/O。 |
| 输入 | 版本化文本、Tokenizer 和固定长度训练块；真实路径按资产 ID 加载 Dataset 与 Tokenizer 当前文件。 |
| 交付物 | 训练预算清单、训练前后验证指标、完整训练状态 Checkpoint 与 Trainer 配置。 |

### 1.1．学习目标

完成本章后，读者能够以 `DatasetDict` 表达训练与验证数据契约，将连续文本转换为 Next-Token 训练块，实现并验证 Decoder-only 训练、AdamW 与学习率调度，恢复完整训练状态，并将原理闭环映射到使用真实语料和标准 Tokenizer 的 Hugging Face `Trainer`。

## 2．直觉与输入输出契约

### 2.1．训练系统的数据流与状态流

训练既有从语料流向参数的数据流，也有从一次更新延续到下一次更新的状态流。任何一条链路缺失，训练都可能无法复现或恢复。

```mermaid
flowchart LR
    A["文本与数据版本"] --> B["DatasetDict"]
    B --> C["Tokenizer 编码契约"]
    C --> D["拼接与定长切块"]
    D --> E["DataLoader"]
    E --> F["Decoder-only 前向传播"]
    F --> G["Next-token Loss"]
    G --> H["反向传播"]
    H --> I["AdamW 更新"]
    I --> J["Warmup + Cosine"]
    J --> E
    I --> K["Checkpoint"]
    K -.->|"恢复模型、优化器、调度器与数据位置"| E
```

第 2～4 节使用规模极小且状态可观察的语料与模型解释闭环；第 5 节把同一契约迁移到固定版本的真实公开语料、正式 Tokenizer 和标准模型。参数量、上下文长度或设备数量扩大后，数据契约、目标函数和恢复语义仍保持不变。

### 2.2．训练阶段及其交付资产

语言模型训练包含多个阶段。各阶段由**起始权重、数据监督信号、目标函数与交付资产**共同界定，而非由是否调用 `backward()` 区分。

| 阶段 | 起点 | 主要数据 | 典型目标 | 产物 |
|---|---|---|---|---|
| 预训练（Pretraining） | 随机初始化 | 大规模通用文本 | Next-token prediction | 基座模型 |
| 持续预训练（Continued Pretraining） | 已有基座模型 | 领域或新增时效语料 | Next-token prediction | 领域增强基座 |
| 监督微调（Supervised Fine-tuning） | 基座模型 | 指令—回答或任务标签 | 只监督目标回答或任务输出 | 指令 / 任务模型 |
| 偏好对齐（Preference Alignment） | 指令模型 | 成对偏好、排序或奖励信号 | DPO、奖励优化或强化学习目标 | 对齐模型 |

本章说明前两类训练的契约，但代码只执行**随机初始化的预训练链路**。持续预训练必须从已版本化的基座权重开始，不能把随机初始化实验改名为持续预训练。两者都优化 next-token 目标；监督微调则通常还会改变模板、Label Mask 和验收指标。

In [ ]:
# 若当前环境缺少依赖，可取消下一行注释并执行；版本以项目 requirements.txt 为准。
# %pip install -U datasets transformers accelerate

import math
import random
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader

# 固定初始化、数据顺序与验证抽样；质量比较需使用预先登记的多个种子。
SEED = 42  # 仅支持实验重放，不保证跨设备或版本逐位一致。
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
device


### 2.3．用 `DatasetDict` 固化语料边界

模型训练消费的不是若干临时字符串，而是具有明确 Split、字段和版本的数据资产。本节直接构造边界已经确定的 `train` 与 `validation`，避免将随机划分逻辑混入训练循环。

- **输入**：包含 `document_id`、`text`、`source` 的文档记录；
- **输出**：`DatasetDict({"train": ..., "validation": ...})`；
- **设计原因**：训练代码只依赖稳定字段，数据采集、清洗、去重与划分由数据工程阶段负责；
- **生产契约**：同时记录数据集 ID、许可证、过滤规则、样本计数与内容摘要，Checkpoint 只记录对这些资产的引用。

验证集不参与参数更新。它回答的是“当前参数在固定留出分布上的 next-token 预测是否改善”，而不是用于反复挑选到对验证集过拟合。


In [ ]:
# 使用结构化文档构造明确的训练集与验证集；小语料只用于观察完整训练机制。
train_records = [
    {"document_id": "train-001", "source": "science", "text": "恒星内部的高温高压使轻元素发生聚变，并持续释放能量。"},
    {"document_id": "train-002", "source": "science", "text": "水循环包括蒸发、凝结、降水和地表径流等相互连接的过程。"},
    {"document_id": "train-003", "source": "math", "text": "矩阵乘法把一组线性变换组合成新的线性变换。"},
    {"document_id": "train-004", "source": "math", "text": "梯度描述函数在当前位置增长最快的方向。"},
    {"document_id": "train-005", "source": "computing", "text": "缓存通过保存重复计算的结果来缩短后续请求的等待时间。"},
    {"document_id": "train-006", "source": "computing", "text": "分布式系统需要处理节点故障、网络延迟和数据一致性。"},
    {"document_id": "train-007", "source": "language", "text": "语言模型根据已有上下文估计下一个符号出现的概率。"},
    {"document_id": "train-008", "source": "language", "text": "词语的含义既来自自身，也来自它与上下文中其他词语的关系。"},
    {"document_id": "train-009", "source": "engineering", "text": "可恢复训练必须同时保存参数、优化器状态和数据消费位置。"},
    {"document_id": "train-010", "source": "engineering", "text": "监控训练损失时还应记录有效 token 数、吞吐率和学习率。"},
]
validation_records = [
    {"document_id": "validation-001", "source": "science", "text": "植物通过光合作用把光能转化为化学能。"},
    {"document_id": "validation-002", "source": "computing", "text": "检查点使中断后的训练能够延续原来的更新轨迹。"},
    {"document_id": "validation-003", "source": "language", "text": "因果注意力只允许当前位置读取它之前的 token。"},
]

corpus = DatasetDict(
    {
        "train": Dataset.from_list(train_records),
        "validation": Dataset.from_list(validation_records),
    }
)
corpus


### 2.4．Tokenizer 输入协议

同一段文本只有在同一套规范化、预分词、词表、特殊 token 与版本下，才会得到相同 token ID。训练资产至少要绑定：

1. Tokenizer 文件及其内容哈希；
2. `pad/bos/eos/unk` 的语义和 ID；
3. 文档连接规则、最大长度和截断方向；
4. 模型 `vocab_size` 与 Tokenizer 词表大小的一致关系。

为保持章节独立性并完整观察 token 变换，本章实现确定性的字符级 `MyCharTokenizer`。该实现用于说明输入协议；生产训练使用与模型资产共同版本化的 Hugging Face fast tokenizer。


In [ ]:
# 从零实现最小 Tokenizer：特殊 Token 先占用固定 ID，其余字符按排序结果建立词表。
class MyCharTokenizer:
    """从训练文本构建带特殊符号的字符级词表，并提供可选特殊符号的编解码。"""
    def __init__(self, texts: list[str]):
        """按排序后的训练字符建立正反向词表，并固定 PAD、BOS、EOS 与 UNK 的 ID。"""
        special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
        characters = sorted(set("".join(texts)))
        self.id_to_token = special_tokens + characters
        self.token_to_id = {token: index for index, token in enumerate(self.id_to_token)}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    @property
    def vocab_size(self) -> int:
        """返回包含特殊符号在内的词表大小。"""
        return len(self.id_to_token)

    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        """将字符串映射为字符 Token ID，并可在两端添加 BOS 与 EOS。"""
        token_ids = [self.token_to_id.get(character, self.unk_token_id) for character in text]
        if add_special_tokens:
            token_ids = [self.bos_token_id, *token_ids, self.eos_token_id]
        return token_ids

    def decode(self, token_ids: list[int], skip_special_tokens: bool = True) -> str:
        """将 Token ID 序列还原为字符串，并可跳过全部特殊符号。"""
        special_ids = {self.pad_token_id, self.bos_token_id, self.eos_token_id, self.unk_token_id}
        tokens = [
            self.id_to_token[token_id]
            for token_id in token_ids
            if not (skip_special_tokens and token_id in special_ids)
        ]
        return "".join(tokens)


# 词表只从训练 split 学习，验证文本只能通过既有词表编码。
training_texts = corpus["train"]["text"]
tokenizer = MyCharTokenizer(training_texts)
encoded_example = tokenizer.encode(corpus["train"][0]["text"])
{"vocab_size": tokenizer.vocab_size, "token_ids": encoded_example[:12], "decoded": tokenizer.decode(encoded_example)}


### 2.5．从文档流构造 Next-Token 训练块

给定 token 序列 $x_0, x_1, \ldots, x_T$，因果语言模型最小化：

$$
\mathcal{L}(\theta) = -\frac{1}{T}\sum_{t=0}^{T-1}\log p_\theta(x_{t+1} \mid x_{\le t})
$$

数据集中令 `labels` 与 `input_ids` 相同，模型内部再把 logits 去掉最后一位、labels 去掉第一位完成移位。这样第 $t$ 个位置的输出预测第 $t+1$ 个 token。

本章采用固定长度 Packing：每篇文档以 BOS 开始、EOS 结束，随后连接成 Token 流并切成等长块。固定块不需要 Padding，计算利用率易于核对。生产管线还需要明确：

- 尾部不足一个块的 token 是丢弃、跨分片续接还是 Padding；
- 是否允许不同文档相互注意；若不允许，需要 block-diagonal attention mask；
- 数据分片、节点数量或恢复位置改变后，token 顺序是否仍可复现。


In [ ]:
# 将每个 split 独立编码和切块，避免训练文本进入验证 token 流。
def my_pack_documents(split: Dataset, tokenizer: MyCharTokenizer, block_size: int) -> Dataset:
    """独立编码一个数据 Split，串接 Token 后切为固定长度块，返回含输入、掩码和标签的 Dataset。"""
    token_stream = []
    for text in split["text"]:
        token_stream.extend(tokenizer.encode(text))

    usable_length = len(token_stream) // block_size * block_size
    blocks = [
        token_stream[start : start + block_size]
        for start in range(0, usable_length, block_size)
    ]
    return Dataset.from_dict(
        {
            "input_ids": blocks,
            "attention_mask": [[1] * block_size for _ in blocks],
            "labels": [block.copy() for block in blocks],
        }
    )


BLOCK_SIZE = 32  # 原理路径的固定 Token 长度；只验证机制，提高后会增加 Attention 与激活成本。
packed_corpus = DatasetDict(
    {split_name: my_pack_documents(split, tokenizer, BLOCK_SIZE) for split_name, split in corpus.items()}
)
packed_corpus


#### 2.5.1．数据模块的职责关系

Dataset、Tokenizer、Packing 和 Collator 各自只负责一种变化。分开这些职责，才能单独审计 token 数量、边界和批次形状。

<!-- diagram:pretraining-data-modules -->

![架构图：预训练数据从文档到批次张量的模块职责与形状](assets/figures/40_pre_training/pretraining-data-modules.svg)

[TikZ 源文件](assets/figures/40_pre_training/pretraining-data-modules.tex)


In [ ]:
# Collator 只做批次堆叠；固定长度切块已经消除了动态 Padding。
def my_collate(records: list[dict]) -> dict[str, torch.Tensor]:
    """把等长记录的 input_ids、attention_mask 和 labels 堆叠成长整型批张量。"""
    return {
        key: torch.tensor([record[key] for record in records], dtype=torch.long)
        for key in ("input_ids", "attention_mask", "labels")
    }


BATCH_SIZE = 2  # 原理路径的微批大小；改变后需按有效 Token 数联动复核学习率与显存。
data_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    packed_corpus["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=data_generator,
    collate_fn=my_collate,
)
validation_loader = DataLoader(
    packed_corpus["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=my_collate,
)
next(iter(train_loader))["input_ids"].shape


<!-- theory-math-contract:v1 -->
### 2.6．核心机制的语言与数学表达

因果语言模型通过最小化有效 Token 上的负对数似然学习 Next-token 条件分布：

$$
\mathcal L_{\mathrm{CLM}}=-\frac{1}{N_{\mathrm{valid}}}
\sum_{b=1}^{B}\sum_{t=1}^{L-1}m_{b,t}\log p_\theta(x_{b,t+1}\mid x_{b,\le t}),
\qquad N_{\mathrm{valid}}=\sum_{b,t}m_{b,t}
$$

其中，$B$ 是 Micro-batch 大小，$L$ 是块长度，$m_{b,t}\in\{0,1\}$ 标记参与损失的 Token，$p_\theta$ 来自 $[B,L,V]$ Logits 的 Softmax。代码中的 `labels == -100` 对应 $m=0$；`CrossEntropyLoss` 实现 Log-Softmax 与负对数似然。Loss 只在固定 Tokenizer、数据版本、Mask 与 Reduction 口径下可比较。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．最小 Decoder-Only 因果语言模型

模型把 token embedding 与 position embedding 相加，依次通过带因果掩码的 Pre-Norm Transformer Block，最后映射到整个词表的 logits。因果掩码保证位置 $t$ 不能读取未来位置。

<!-- diagram:pretraining-model-architecture -->

![架构图：Decoder-only 预训练模型的嵌入、因果块与词表投影架构](assets/figures/40_pre_training/pretraining-model-architecture.svg)

[TikZ 源文件](assets/figures/40_pre_training/pretraining-model-architecture.tex)

训练时，`labels [B,T]` 与 logits 一起进入 `Shift + Cross Entropy`；标签不是模型前向结构的一部分。

本节从零实现训练所需的模型外壳与残差结构，多头注意力和线性层仍使用 PyTorch 稳定算子；注意力内部推导见模型架构章节。


In [ ]:
# 64 维、4 头、2 层与两倍 FFN 控制原理实现规模；头数变化后仍须满足整除。
from tqdm.auto import tqdm, trange

@dataclass
class MyModelConfig:
    """集中定义最小因果语言模型的词表、上下文、网络宽深度与数值配置。"""
    vocab_size: int
    max_sequence_length: int
    hidden_size: int = 64
    num_heads: int = 4
    num_layers: int = 2
    intermediate_size: int = 128
    dropout: float = 0.0
    layer_norm_epsilon: float = 1e-5


# Pre-Norm Block：归一化后分别进入注意力和 MLP，再通过残差连接回主干。
class MyCausalBlock(nn.Module):
    """实现 Pre-Norm 因果自注意力与前馈网络组成的残差 Transformer Block。"""
    def __init__(self, config: MyModelConfig):
        """根据模型配置创建归一化、多头注意力和 MLP 子层。"""
        super().__init__()
        self.attention_norm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_epsilon
        )
        self.attention = nn.MultiheadAttention(
            config.hidden_size, config.num_heads, dropout=config.dropout, batch_first=True
        )
        self.mlp_norm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_epsilon
        )
        self.mlp = nn.Sequential(
            nn.Linear(config.hidden_size, config.intermediate_size),
            nn.GELU(),
            nn.Linear(config.intermediate_size, config.hidden_size),
            nn.Dropout(config.dropout),
        )

    def forward(
        self,
        hidden_states: torch.Tensor,
        causal_mask: torch.Tensor,
        padding_mask: torch.Tensor | None,
    ) -> torch.Tensor:
        """对 [B,T,D] 隐状态应用因果与 Padding Mask，再完成注意力和 MLP 两次残差更新。"""
        normalized_states = self.attention_norm(hidden_states)
        attention_output, _ = self.attention(
            normalized_states,
            normalized_states,
            normalized_states,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        hidden_states = hidden_states + attention_output
        hidden_states = hidden_states + self.mlp(self.mlp_norm(hidden_states))
        return hidden_states


In [ ]:
# -100 是交叉熵的标签屏蔽哨兵，不是 Token ID；加入 Padding 时必须同步数据与损失协议。
LABEL_IGNORE_INDEX = -100

# 模型内部完成因果掩码、logits 生成和 labels 移位，接口与 Trainer 约定保持一致。
class MyCausalLM(nn.Module):
    """实现字符级 Decoder-only 语言模型，内部构造因果掩码并支持移位语言建模损失。"""
    def __init__(self, config: MyModelConfig):
        """创建 Token/位置嵌入、堆叠 Block、最终归一化和权重共享输出头。"""
        super().__init__()
        self.config = config
        self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_sequence_length, config.hidden_size)
        self.blocks = nn.ModuleList([MyCausalBlock(config) for _ in range(config.num_layers)])
        self.final_norm = nn.LayerNorm(
            config.hidden_size, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embeddings.weight

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **unused,
    ) -> dict[str, torch.Tensor]:
        """把 [B,T] 输入映射为 [B,T,V] logits，并在提供 labels 时返回忽略屏蔽位的移位交叉熵。"""
        _, sequence_length = input_ids.shape
        position_ids = torch.arange(sequence_length, device=input_ids.device)
        hidden_states = self.token_embeddings(input_ids) + self.position_embeddings(position_ids)

        # True 表示不可见位置；Padding Mask 则屏蔽批次中补齐的键。
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )
        padding_mask = attention_mask.eq(0) if attention_mask is not None else None
        for block in self.blocks:
            hidden_states = block(hidden_states, causal_mask, padding_mask)

        logits = self.lm_head(self.final_norm(hidden_states))
        outputs = {"logits": logits}
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            outputs["loss"] = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=LABEL_IGNORE_INDEX,
            )
        return outputs


config = MyModelConfig(vocab_size=tokenizer.vocab_size, max_sequence_length=BLOCK_SIZE)
model = MyCausalLM(config).to(device)


In [ ]:
# 用一个批次观察接口形状；loss 已在模型内部完成 next-token 移位。
sample_batch = {key: value.to(device) for key, value in next(iter(train_loader)).items()}
sample_outputs = model(**sample_batch)
{
    "input_shape": tuple(sample_batch["input_ids"].shape),
    "logits_shape": tuple(sample_outputs["logits"].shape),
    "loss": sample_outputs["loss"].item(),
}


#### 3.1.1．机制可视化：移位目标与逐位置损失

**学习问题**：长度为 $T$ 的输入为何只形成 $T-1$ 个监督目标，且批次平均损失由哪些位置贡献？

**验收不变量**：图中同一横坐标 $t$ 上方是上下文末端 token $x_t$，下方是该位置 logits 对应的监督目标 $x_{t+1}$；`shift_logits`、`shift_labels` 与逐位置损失的序列长度均为 `BLOCK_SIZE - 1`。柱形高度直接由当前 `sample_outputs["logits"]` 和 `sample_batch["labels"]` 计算，不另造样本或损失。


In [ ]:
import matplotlib.pyplot as plt

shift_logits_for_plot = sample_outputs["logits"][:, :-1, :].detach()
shift_labels_for_plot = sample_batch["labels"][:, 1:].detach()
if shift_logits_for_plot.shape[:2] != shift_labels_for_plot.shape:
    raise RuntimeError("移位后的 logits 与 labels 位置维度不一致")
if shift_labels_for_plot.shape[1] != BLOCK_SIZE - 1:
    raise RuntimeError("固定长度训练块应产生 BLOCK_SIZE - 1 个 next-token 目标")

flat_position_losses = F.cross_entropy(
    shift_logits_for_plot.reshape(-1, config.vocab_size),
    shift_labels_for_plot.reshape(-1),
    ignore_index=LABEL_IGNORE_INDEX,
    reduction="none",
).reshape_as(shift_labels_for_plot)
valid_target_mask = shift_labels_for_plot.ne(LABEL_IGNORE_INDEX)
valid_counts_by_position = valid_target_mask.sum(dim=0)
if torch.any(valid_counts_by_position == 0):
    raise RuntimeError("存在没有有效监督目标的位置，无法绘制逐位置损失")
mean_loss_by_position = (
    (flat_position_losses * valid_target_mask).sum(dim=0) / valid_counts_by_position
).cpu()

example_input_ids = sample_batch["input_ids"][0].detach().cpu().tolist()
context_tokens = [tokenizer.id_to_token[token_id] for token_id in example_input_ids[:-1]]
target_tokens = [tokenizer.id_to_token[token_id] for token_id in example_input_ids[1:]]
positions = list(range(BLOCK_SIZE - 1))

figure, (shift_axis, loss_axis) = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
shift_axis.scatter(positions, [1] * len(positions), color="#2563eb", s=24, label=r"上下文末端 $x_t$")
shift_axis.scatter(positions, [0] * len(positions), color="#dc2626", s=24, label=r"监督目标 $x_{t+1}$")
for position, context_token, target_token in zip(positions, context_tokens, target_tokens):
    shift_axis.annotate("", xy=(position, 0.15), xytext=(position, 0.85), arrowprops={"arrowstyle": "->", "color": "#64748b", "lw": 0.8})
    shift_axis.text(position, 1.08, context_token, ha="center", va="bottom", fontsize=8)
    shift_axis.text(position, -0.08, target_token, ha="center", va="top", fontsize=8)
shift_axis.set(yticks=[0, 1], yticklabels=[r"label $x_{t+1}$", r"input $x_t$"], xlim=(-0.6, BLOCK_SIZE - 1.4))
shift_axis.set_title("同一 logit 位置上的输入—目标移位")
shift_axis.set_xlabel(r"logit 位置 $t$")
shift_axis.grid(axis="x", alpha=0.2)

loss_axis.bar(positions, mean_loss_by_position.tolist(), color="#7c3aed", alpha=0.82)
loss_axis.axhline(float(sample_outputs["loss"].detach().cpu()), color="#111827", linestyle="--", label="批次平均 loss")
loss_axis.set(title="当前批次逐位置交叉熵", xlabel=r"目标位置 $t+1$", ylabel="negative log-likelihood")
loss_axis.grid(axis="y", alpha=0.25)
loss_axis.legend()
plt.show()


**应观察结论**：第一个输入 token 监督第二个 token，倒数第二个输入 token 监督最后一个 token；最后一个位置的 logits 没有块内后继目标，因而不进入损失。不同位置的柱形高度通常不同，批次 `loss` 是全部有效位置负对数似然的平均，而不是某个位置的损失。

**不可误读边界**：单个小批次的高损失位置只说明当前参数对该位置目标分配的概率较低，不能据此判定某类语言现象更难，也不能把 token 位置与字符位置等同。跨文档 Packing、Padding、`ignore_index` 或 Tokenizer 变化都会改变有效位置集合。


### 3.2．AdamW、Warmup 与 Cosine Decay

参数更新同时受优化器、参数分组和学习率调度影响。本节建立三个基本约定：

- **AdamW** 将权重衰减与梯度更新解耦；矩阵参数参与衰减，bias 与归一化参数不衰减；
- **Warmup** 在训练初期逐步升高学习率，避免随机初始化时过大的更新；
- **Cosine Decay** 在剩余步数中平滑降低学习率。

设峰值学习率为 $\eta_{max}$、Warmup 步数为 $W$、总更新步数为 $S$，本章使用：

$$
\eta_s = \eta_{max}
\begin{cases}
\frac{s + 1}{W}, & s < W \\
\frac{1}{2}\left[1 + \cos\left(\pi\frac{s-W}{S-W}\right)\right], & W \le s < S
\end{cases}
$$

调度器按**参数更新步**推进，而不是按读取的 micro-batch 数推进。梯度累积会改变两者关系，相关细节放在训练优化章节。峰值学习率应在相同有效 token 预算下按对数尺度比较：初期 loss 或梯度范数尖峰通常说明取值过大，稳定但几乎不下降则可能过小。Warmup 步数随总更新预算确定；本章的绝对值 `2` 仅用于展示调度阶段。


In [ ]:
# 二维及以上权重参与衰减；向量参数通常是 bias 或归一化参数。
ADAM_BETAS = (0.9, 0.95)  # 较低 beta2 更快响应梯度尺度变化；版本化配置必须显式记录。
ADAM_EPSILON = 1e-8  # AdamW 分母的数值稳定项。
MAX_GRAD_NORM = 1.0  # 全局梯度范数上限；频繁裁剪时应排查学习率、Batch 与异常样本。

def my_build_optimizer(model: nn.Module, learning_rate: float, weight_decay: float):
    """按参数维度拆分衰减与不衰减组，并创建使用固定 betas 和 epsilon 的 AdamW。"""
    decay_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.dim() >= 2]
    no_decay_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad and parameter.dim() < 2]
    parameter_groups = [
        {"params": decay_parameters, "weight_decay": weight_decay},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(
        parameter_groups, lr=learning_rate, betas=ADAM_BETAS, eps=ADAM_EPSILON
    )


def my_lr_multiplier(step: int, warmup_steps: int, total_steps: int) -> float:
    """计算线性 Warmup 后余弦衰减的学习率倍率。"""
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


LEARNING_RATE = 3e-4  # Decoder-only 的起始学习率；模型、有效 Batch 或精度变化后需重新扫描。
WEIGHT_DECAY = 0.1  # 当前正则起点；应在固定 Token 预算下比较验证损失。
NUM_EPOCHS = 2  # 原理路径仅产生有限更新，不用于评价预训练质量。
WARMUP_STEPS = 2  # 以当前总步数展示 Warmup；预算变化后必须重新核算占比。
TOTAL_STEPS = len(train_loader) * NUM_EPOCHS
if not 0 < WARMUP_STEPS < TOTAL_STEPS:
    raise ValueError("warmup steps 必须严格位于 0 与 total steps 之间")

optimizer = my_build_optimizer(model, LEARNING_RATE, WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: my_lr_multiplier(step, WARMUP_STEPS, TOTAL_STEPS),
)


## 4．证据验证

### 4.1．训练与验证闭环

一次参数更新的顺序是：清空旧梯度 → 前向传播 → 计算 loss → 反向传播 → 裁剪梯度 → AdamW 更新 → 调度器推进。验证阶段使用 `eval()` 和 `no_grad()`，不写入梯度。

评估聚合的是所有有效目标 token 的负对数似然，而不是简单平均各批次 loss。Perplexity（困惑度）定义为：

$$
\operatorname{PPL} = \exp\left(\frac{\sum_i \operatorname{NLL}_i}{\sum_i N_i}\right)
$$

其中 $N_i$ 是第 $i$ 个批次参与 next-token loss 的 token 数。PPL 只适合在相同 Tokenizer、相同验证集和相同边界规则下比较。


In [ ]:
@dataclass
class MyTrainState:
    """记录可恢复训练的全局步数、完成 Epoch、已见 Token 与最佳验证损失。"""
    global_step: int = 0
    completed_epochs: int = 0
    tokens_seen: int = 0
    best_validation_loss: float = math.inf


# 验证按有效目标 token 加权汇总，避免最后一个小批次改变整体权重。
@torch.no_grad()
def my_evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    """在无梯度模式下按有效目标 Token 加权验证损失，并返回 Loss 与 Perplexity。"""
    model.eval()
    total_negative_log_likelihood = 0.0
    total_target_tokens = 0

    for batch in tqdm(
        loader, desc="验证最小 Causal LM", unit="batch", leave=False, dynamic_ncols=True
    ):
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**batch)
        target_tokens = int(batch["attention_mask"][:, 1:].sum().item())
        total_negative_log_likelihood += outputs["loss"].item() * target_tokens
        total_target_tokens += target_tokens

    mean_loss = total_negative_log_likelihood / total_target_tokens
    return {"loss": mean_loss, "perplexity": math.exp(mean_loss)}


# 一个 epoch 内每个批次完成一次参数更新，并同步推进学习率与 token 计数。
def my_train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    state: MyTrainState,
    device: torch.device,
    description: str,
) -> float:
    """遍历一个 Epoch 完成反向传播、梯度裁剪、优化器与调度器更新，并原地推进训练状态。"""
    model.train()
    total_loss = 0.0

    for batch in tqdm(
        loader, desc=description, unit="batch", leave=False, dynamic_ncols=True
    ):
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        state.global_step += 1
        state.tokens_seen += int(batch["attention_mask"][:, 1:].sum().item())
        total_loss += outputs["loss"].item()

    return total_loss / len(loader)


In [ ]:
# 每个 epoch 结束后在固定验证集上评估，并记录参数更新步与有效 token 数。
state = MyTrainState()
history = []
epoch_progress = trange(
    NUM_EPOCHS, desc="预训练最小 Causal LM", unit="epoch", dynamic_ncols=True
)
for epoch in epoch_progress:
    train_loss = my_train_epoch(
        model, train_loader, optimizer, scheduler, state, device,
        description=f"Epoch {epoch + 1} 训练",
    )
    validation_metrics = my_evaluate(model, validation_loader, device)
    state.completed_epochs = epoch + 1
    state.best_validation_loss = min(state.best_validation_loss, validation_metrics["loss"])
    epoch_progress.set_postfix(
        train_loss=f"{train_loss:.4f}",
        validation_loss=f"{validation_metrics['loss']:.4f}",
        lr=f"{scheduler.get_last_lr()[0]:.2e}",
    )
    history.append(
        {
            "epoch": state.completed_epochs,
            "global_step": state.global_step,
            "tokens_seen": state.tokens_seen,
            "train_loss": train_loss,
            "validation_loss": validation_metrics["loss"],
            "validation_perplexity": validation_metrics["perplexity"],
            "learning_rate": scheduler.get_last_lr()[0],
        }
    )
history


### 4.2．Checkpoint 恢复一致性

只保存 `model.state_dict()` 适合推理，不足以精确继续训练。可恢复 Checkpoint 至少要覆盖：

- 模型参数与 buffer；
- AdamW 的一阶、二阶矩和参数组；
- 学习率调度器当前位置；
- `global_step`、已完成 epoch、已消费 token 数；
- 数据采样器 / 生成器状态与数据游标；
- Python、PyTorch 及每个训练进程的随机状态；
- 模型配置、Tokenizer 文件哈希、数据版本、代码与依赖版本。

```mermaid
sequenceDiagram
    participant T as Training Loop
    participant S as Sampler / RNG
    participant C as Checkpoint Store
    participant R as Resumed Process
    T->>T: 完成一次 optimizer.step
    T->>S: 读取数据位置与随机状态
    T->>C: 原子提交完整训练状态
    C-->>R: 加载已保存的资产与状态
    R->>S: 恢复采样顺序和随机状态
    R->>R: 从下一次参数更新继续
```

本章在 epoch 边界保存，因此数据位置由 `completed_epochs` 和 DataLoader generator 状态共同表达。生产中的中途恢复还需保存当前 shard、sampler epoch、batch offset，以及每个 rank 的随机状态；变更 world size 时必须明确是否接受数据顺序改变。


In [ ]:
# Checkpoint 由可信训练进程写入；torch.save 的优化器状态不应从不可信来源加载。
def my_save_checkpoint(
    checkpoint_path: Path,
    model: MyCausalLM,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    state: MyTrainState,
    data_generator: torch.Generator,
    tokenizer: MyCharTokenizer,
) -> None:
    """将模型、优化器、调度器、训练状态、词表及随机状态原子写入可信 Checkpoint 路径。"""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    training_state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "train_state": asdict(state),
        "model_config": asdict(model.config),
        "tokenizer_vocab": tokenizer.id_to_token,
        "python_rng_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "data_generator_state": data_generator.get_state(),
    }
    temporary_path = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
    torch.save(training_state, temporary_path)
    temporary_path.replace(checkpoint_path)


def my_load_checkpoint(
    checkpoint_path: Path,
    model: MyCausalLM,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    data_generator: torch.Generator,
) -> MyTrainState:
    """从可信 Checkpoint 恢复模型、优化器、调度器和随机状态，并返回训练进度。"""
    training_state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(training_state["model"])
    optimizer.load_state_dict(training_state["optimizer"])
    scheduler.load_state_dict(training_state["scheduler"])
    random.setstate(training_state["python_rng_state"])
    torch.set_rng_state(training_state["torch_rng_state"])
    data_generator.set_state(training_state["data_generator_state"])
    return MyTrainState(**training_state["train_state"])


In [ ]:
# 保存 epoch 边界状态，再构造全新的模型与优化器恢复；自然暴露任何资产不一致。
checkpoint_path = Path(tempfile.mkdtemp(prefix="model-training-")) / "checkpoint.pt"
my_save_checkpoint(checkpoint_path, model, optimizer, scheduler, state, data_generator, tokenizer)

resumed_model = MyCausalLM(config).to(device)
resumed_optimizer = my_build_optimizer(resumed_model, LEARNING_RATE, WEIGHT_DECAY)
resumed_scheduler = torch.optim.lr_scheduler.LambdaLR(
    resumed_optimizer,
    lr_lambda=lambda step: my_lr_multiplier(step, WARMUP_STEPS, TOTAL_STEPS),
)
resumed_generator = torch.Generator()
resumed_state = my_load_checkpoint(
    checkpoint_path,
    resumed_model,
    resumed_optimizer,
    resumed_scheduler,
    resumed_generator,
)
asdict(resumed_state)


## 5．迁移到生产库

### 5.1．Hugging Face `Trainer` 职责映射

原理训练闭环建立了可解释基线，`Trainer` 将同一职责标准化。使用生产可用组件不等于已经完成生产规模预训练；数据治理、训练预算、容量、监控和分布式恢复仍需单独验收。

| 原理实现组件 | 第 5 节标准实现 |
|---|---|
| 原理构造的 `corpus` | 仓库当前提供的 WikiText-2 `train` / `validation` |
| `MyCharTokenizer` | 仓库当前提供的 GPT-2 Fast Tokenizer |
| `my_pack_documents` | `Dataset.map()` 批量编码、追加 EOS、有界缓冲定长 Packing |
| `MyCausalLM.forward(..., labels=...)` | `AutoModelForCausalLM.from_config()` 随机初始化的 GPT-2 Causal LM |
| AdamW、调度器、裁剪 | `TrainingArguments` 中的 AdamW、Warmup + Cosine 与 clip norm |
| `my_evaluate` | 训练前基线、步进评估与训练后 `Trainer.evaluate()` |
| 完整状态恢复 | `Trainer` Checkpoint + 数据 / Tokenizer / 配置 Manifest；恢复前校验契约 |

真实路径按以下资产 ID 加载上游当前文件，并在本次运行中记录实际文件 SHA-256：

- Dataset：`Salesforce/wikitext` / `wikitext-2-raw-v1`，许可证 CC BY-SA 3.0 / GFDL；
- Tokenizer：`openai-community/gpt2`，Fast byte-level BPE；
- Model：仅复用 GPT-2 的结构配置，权重由 `from_config()` 随机初始化，**没有加载预训练权重**。

本实验使用完整训练 split、固定验证子集并封存 test split。首次执行需要联网下载约数 MB 的上游资产；失败时应显式报错，不能静默回退到人工构造语料。

### 5.2．加载固定版本的真实语料与 Tokenizer

WikiText 的每个非空 `text` 行在这里视为一个**文本段**，而不是声称它一定是一篇完整文章。每段末尾追加 EOS 后再连接；`test` split 不进入任何预处理、调参或验收。GPT-2 没有独立 PAD token，而本实验全部是等长块，因此将 PAD 角色指向 EOS 只为满足标准接口，不产生 padding token。

In [ ]:
import hashlib

from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

# 加载 WikiText-2 当前提供的 train/validation 资产；该小语料只用于可审计训练，不代表生产配方。
REAL_DATASET_ID = "Salesforce/wikitext"
REAL_DATASET_CONFIG = "wikitext-2-raw-v1"
REAL_DATA_FILES = {
    "train": "wikitext-2-raw-v1/train-00000-of-00001.parquet",
    "validation": "wikitext-2-raw-v1/validation-00000-of-00001.parquet",
}
# 加载 GPT-2 Fast Tokenizer 当前文件；其 50,257 个 ID 是资产契约，替换后必须重新切块并重建模型。
REAL_TOKENIZER_ID = "openai-community/gpt2"


def calculate_file_sha256(file_path: str | Path) -> str:
    """流式读取文件并返回其 SHA-256 十六进制摘要。"""
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# 只下载 train / validation 文件；逐文件计算并记录内容哈希，test 不下载。
real_local_data_files = {
    split_name: hf_hub_download(
        repo_id=REAL_DATASET_ID,
        repo_type="dataset",
        filename=repo_path,
        )
    for split_name, repo_path in REAL_DATA_FILES.items()
}
real_actual_file_sha256 = {
    split_name: calculate_file_sha256(file_path)
    for split_name, file_path in real_local_data_files.items()
}
real_raw_dataset = load_dataset("parquet", data_files=real_local_data_files)


def has_nonempty_text(record: dict) -> bool:
    """判断数据记录的 text 字段去除空白后是否仍有内容。"""
    return bool(record["text"].strip())


real_corpus = DatasetDict(
    {
        split_name: real_raw_dataset[split_name].filter(
            has_nonempty_text,
            desc=f"过滤 {split_name} 空文本段",
        )
        for split_name in ("train", "validation")
    }
)
real_tokenizer = AutoTokenizer.from_pretrained(
    REAL_TOKENIZER_ID,
    use_fast=True,
)
if real_tokenizer.pad_token_id is None:
    real_tokenizer.pad_token = real_tokenizer.eos_token

if not real_tokenizer.is_fast:
    raise TypeError("真实训练路径要求 Hugging Face Fast Tokenizer。")
if len(real_tokenizer) != 50_257:
    raise ValueError(f"GPT-2 Tokenizer 的词表应为 50,257，实际为 {len(real_tokenizer):,}。")
if real_tokenizer.eos_token_id != 50_256:
    raise ValueError(f"GPT-2 Tokenizer 的 EOS ID 应为 50,256，实际为 {real_tokenizer.eos_token_id}。")
if real_tokenizer.pad_token_id != real_tokenizer.eos_token_id:
    raise ValueError("等长 Packing 路径要求 PAD 角色显式复用 EOS。")

real_asset_summary = {
    "dataset": f"{REAL_DATASET_ID}/{REAL_DATASET_CONFIG}",
    "dataset_license": ["CC-BY-SA-3.0", "GFDL"],
    "source_files": {
        split_name: {
            "repo_path": REAL_DATA_FILES[split_name],
            "sha256": real_actual_file_sha256[split_name],
            "size_bytes": Path(file_path).stat().st_size,
        }
        for split_name, file_path in real_local_data_files.items()
    },
    "segment_filter": "text.strip() != ''",
    "raw_rows": {name: len(split) for name, split in real_raw_dataset.items()},
    "nonempty_segments": {name: len(split) for name, split in real_corpus.items()},
    "tokenizer": REAL_TOKENIZER_ID,
    "tokenizer_repository_license": "MIT",
    "vocab_size": len(real_tokenizer),
    "test_policy": "sealed; not downloaded or inspected",
}
real_asset_summary

### 5.3．把文本段转换为固定长度训练块

编码阶段不让 Tokenizer 自动添加特殊 token，而是在每个非空文本段末尾显式追加一个 EOS。随后每个 split 各自拼接，并以 `128` token 切块；尾部不足一个块的 token 丢弃且被计入摘要。

Packing 使用跨编码批次保留尾部的有界缓冲：每次最多物化 512 个文本段与 1,024 个待写 block，批次边界不会额外丢 token，也不会同时复制整个 split 的 token stream、labels 和 attention mask。验证集经过 seed `42` 的确定性打乱后固定选择 256 个块；训练集不截断，由 `Trainer` 的 seeded sampler 决定本次预算消费的块。test split 不下载、不预处理，也不参与参数或超参数选择。

In [ ]:
from datasets import concatenate_datasets

REAL_BLOCK_SIZE = 128  # 真实路径的固定 Token 长度；提高会增加 Attention、激活与显存成本。
REAL_EVAL_BLOCK_LIMIT = 256  # 冻结验证子集的块数上限；发布评测必须扩大覆盖并报告不确定性。
REAL_TOKENIZE_BATCH_SIZE = 512  # 仅控制预处理吞吐与内存，不改变 Tokenizer 结果。
REAL_PACKING_WRITE_BLOCKS = 1_024  # 每次物化的块数，用于限制中间内存占用。


def tokenize_real_segments(batch: dict[str, list[str]]) -> dict[str, list[list[int]]]:
    """批量使用固定 GPT-2 Tokenizer 编码文本段，并在每段末尾追加 EOS。"""
    encoded = real_tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )
    eos_token_id = real_tokenizer.eos_token_id
    return {"input_ids": [token_ids + [eos_token_id] for token_ids in encoded["input_ids"]]}


def blocks_to_dataset(blocks: list[list[int]]) -> Dataset:
    """把固定长度 Token 块物化为包含输入、全一掩码与复制标签的 Dataset。"""
    return Dataset.from_dict(
        {
            "input_ids": blocks,
            "attention_mask": [[1] * REAL_BLOCK_SIZE for _ in blocks],
            "labels": [block.copy() for block in blocks],
        }
    )


def pack_real_split(encoded_split: Dataset) -> tuple[Dataset, int]:
    """流式拼接已编码文本并写出固定长度 Dataset 分片，返回合并结果与丢弃的尾部 Token 数。"""
    token_buffer: list[int] = []
    pending_blocks: list[list[int]] = []
    packed_shards: list[Dataset] = []

    for batch in encoded_split.iter(batch_size=REAL_TOKENIZE_BATCH_SIZE):
        for token_ids in batch["input_ids"]:
            token_buffer.extend(token_ids)

        usable_length = len(token_buffer) // REAL_BLOCK_SIZE * REAL_BLOCK_SIZE
        for start in range(0, usable_length, REAL_BLOCK_SIZE):
            pending_blocks.append(token_buffer[start : start + REAL_BLOCK_SIZE])
            if len(pending_blocks) == REAL_PACKING_WRITE_BLOCKS:
                packed_shards.append(blocks_to_dataset(pending_blocks))
                pending_blocks = []
        token_buffer = token_buffer[usable_length:]

    if pending_blocks:
        packed_shards.append(blocks_to_dataset(pending_blocks))
    if not packed_shards:
        raise ValueError("编码后的 split 不足以形成一个完整训练块。")

    return concatenate_datasets(packed_shards), len(token_buffer)


real_encoded_corpus = real_corpus.map(
    tokenize_real_segments,
    batched=True,
    batch_size=REAL_TOKENIZE_BATCH_SIZE,
    remove_columns=["text"],
    desc="GPT-2 Fast Tokenizer 编码",
)
real_token_counts = {
    split_name: sum(
        len(token_ids)
        for batch in split.iter(batch_size=REAL_TOKENIZE_BATCH_SIZE)
        for token_ids in batch["input_ids"]
    )
    for split_name, split in real_encoded_corpus.items()
}

real_packed_datasets: dict[str, Dataset] = {}
real_dropped_tail_tokens: dict[str, int] = {}
for split_name, split in real_encoded_corpus.items():
    packed_split, dropped_tail_tokens = pack_real_split(split)
    real_packed_datasets[split_name] = packed_split
    real_dropped_tail_tokens[split_name] = dropped_tail_tokens
real_packed_full = DatasetDict(real_packed_datasets)
for split_name in ("train", "validation"):
    accounted_tokens = (
        len(real_packed_full[split_name]) * REAL_BLOCK_SIZE
        + real_dropped_tail_tokens[split_name]
    )
    if accounted_tokens != real_token_counts[split_name]:
        raise RuntimeError(
            f"{split_name} Packing 记账不一致：编码后 {real_token_counts[split_name]:,} token，"
            f"切块与尾部合计 {accounted_tokens:,} token。"
        )

validation_size = min(REAL_EVAL_BLOCK_LIMIT, len(real_packed_full["validation"]))
real_packed_corpus = DatasetDict(
    {
        "train": real_packed_full["train"],
        "validation": real_packed_full["validation"]
        .shuffle(seed=SEED)
        .select(range(validation_size)),
    }
)

real_packing_summary = {
    split_name: {
        "tokens_before_packing": real_token_counts[split_name],
        "full_blocks": len(real_packed_full[split_name]),
        "tokens_dropped_at_tail": real_dropped_tail_tokens[split_name],
        "blocks_used": len(real_packed_corpus[split_name]),
        "datasets_cache_fingerprint": real_packed_corpus[split_name]._fingerprint,
    }
    for split_name in ("train", "validation")
}
real_packing_summary

### 5.4．建立可审计的真实训练预算

模型是随机初始化、约 724 万参数的标准 GPT-2 Causal LM。`max_steps=1,500` 覆盖 epoch 设置；单设备每次更新消费 8 个长度为 128 的块，即 1,016 个 next-token 目标，总预算为 1,524,000 个目标 token。每 500 步评估并保存，训练结束恢复验证损失最佳的 Checkpoint。

墙钟时间不是训练契约：同一预算在不同设备上会不同。常见 Colab/Apple 加速器的目标区间约为 1～3 分钟，CPU 通常更久；同步盘 Checkpoint I/O 也可能显著增加时间。运行前应以本单元输出的参数量、global batch、目标 token 和可用数据比例为准，而不是人为延时。本章预算只定义单进程单设备；分布式 global batch 与统一 run ID 进入第 6 节和训练优化章节。

In [ ]:
import json
import os
import platform
from datetime import datetime, timezone

import datasets
import transformers
from transformers import (
    AutoModelForCausalLM,
    GPT2Config,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

REAL_MICRO_BATCH_SIZE = 8  # 单设备微批；与累积步和数据并行数共同决定 Global Batch。
REAL_GRAD_ACCUMULATION_STEPS = 1  # 当前单设备 Global Batch 为 8；改变后需联动复核学习率与 Token 预算。
REAL_MAX_STEPS = 1_500  # 主停止预算，覆盖 epoch 设置；单设备共处理 1,524,000 个目标 Token。
REAL_WARMUP_STEPS = 75  # 占总更新的 5%，用于控制初期更新；总预算变化后需重新核算。
REAL_EVAL_INTERVAL = 500  # 每 500 步评估并保存，共形成 3 个步进验证点。
REAL_LEARNING_RATE = 3e-4  # 当前峰值学习率；模型、Global Batch、精度或数据变化后重新扫描。
REAL_WEIGHT_DECAY = 0.1  # Decoder-only 训练的正则起点，应以冻结验证集复核。
REAL_ADAM_BETAS = (0.9, 0.95)  # 较低 beta2 更快响应梯度尺度变化。
REAL_ADAM_EPSILON = 1e-8  # AdamW 分母的数值稳定项。
REAL_MAX_GRAD_NORM = 1.0  # 全局梯度范数上限；高频触发时应排查训练配置。

# 128 维、4 头、4 层、512 维 FFN与 0.1 Dropout 保留标准 GPT-2 结构并控制单设备成本。
set_seed(SEED)
real_model_config = GPT2Config(
    vocab_size=len(real_tokenizer),
    n_positions=REAL_BLOCK_SIZE,
    n_embd=128,
    n_layer=4,
    n_head=4,
    n_inner=512,
    activation_function="gelu_new",
    resid_pdrop=0.1,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    use_cache=False,
    bos_token_id=real_tokenizer.eos_token_id,
    eos_token_id=real_tokenizer.eos_token_id,
    pad_token_id=real_tokenizer.pad_token_id,
)
real_model = AutoModelForCausalLM.from_config(real_model_config)
# Transformers 5.13 的旧类名 GPT2LMHeadModel 不会自动命中通用 Loss 注册表；显式绑定同一 Causal LM 目标。
real_model.loss_type = "ForCausalLM"
real_model_parameter_count = sum(parameter.numel() for parameter in real_model.parameters())
if real_model_parameter_count != 7_242_624:
    raise RuntimeError(
        f"模型结构与预算不一致：预期 7,242,624 个参数，实际 {real_model_parameter_count:,}。"
    )

REAL_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REAL_ARTIFACT_ROOT_ENV = "LLM_FROM_SCRATCH_ARTIFACT_ROOT"
real_artifact_root = Path(
    os.environ.get(
        REAL_ARTIFACT_ROOT_ENV,
        "artifacts/40_pre_training/gpt2-wikitext2-small",
    )
)
real_output_dir = real_artifact_root / REAL_RUN_ID
real_training_arguments = TrainingArguments(
    output_dir=str(real_output_dir),
    run_name=f"gpt2-wikitext2-small-{REAL_RUN_ID}",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=REAL_MICRO_BATCH_SIZE,
    per_device_eval_batch_size=REAL_MICRO_BATCH_SIZE,
    gradient_accumulation_steps=REAL_GRAD_ACCUMULATION_STEPS,
    max_steps=REAL_MAX_STEPS,
    learning_rate=REAL_LEARNING_RATE,
    weight_decay=REAL_WEIGHT_DECAY,
    adam_beta1=REAL_ADAM_BETAS[0],
    adam_beta2=REAL_ADAM_BETAS[1],
    adam_epsilon=REAL_ADAM_EPSILON,
    max_grad_norm=REAL_MAX_GRAD_NORM,
    warmup_steps=REAL_WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=REAL_EVAL_INTERVAL,
    save_strategy="steps",
    save_steps=REAL_EVAL_INTERVAL,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    dataloader_drop_last=True,
    dataloader_pin_memory=torch.cuda.is_available(),
    fp16=False,
    bf16=False,
    tf32=False,
    save_safetensors=True,
    report_to="none",
    remove_unused_columns=False,
    disable_tqdm=False,
    seed=SEED,
    data_seed=SEED,
)

if real_training_arguments.world_size != 1 or real_training_arguments.n_gpu > 1:
    raise RuntimeError(
        "本章真实训练预算仅支持单进程单设备；分布式训练需要统一 run ID 并重新核算 global batch。"
    )

real_global_batch_size = REAL_MICRO_BATCH_SIZE * REAL_GRAD_ACCUMULATION_STEPS
real_target_token_budget = (
    REAL_MAX_STEPS * real_global_batch_size * (REAL_BLOCK_SIZE - 1)
)
real_train_blocks_per_complete_pass = (
    len(real_packed_corpus["train"]) // real_global_batch_size * real_global_batch_size
)
real_available_train_targets = real_train_blocks_per_complete_pass * (REAL_BLOCK_SIZE - 1)
real_run_budget = {
    "model_parameters": real_model_parameter_count,
    "world_size": real_training_arguments.world_size,
    "micro_batch_size": REAL_MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": REAL_GRAD_ACCUMULATION_STEPS,
    "global_batch_size": real_global_batch_size,
    "block_size": REAL_BLOCK_SIZE,
    "max_steps": REAL_MAX_STEPS,
    "effective_target_tokens": real_target_token_budget,
    "packed_train_blocks": len(real_packed_corpus["train"]),
    "train_blocks_per_complete_pass": real_train_blocks_per_complete_pass,
    "available_train_target_tokens_per_complete_pass": real_available_train_targets,
    "fraction_of_one_data_pass": real_target_token_budget / real_available_train_targets,
    "eval_blocks": len(real_packed_corpus["validation"]),
    "seed": SEED,
    "learning_rate": REAL_LEARNING_RATE,
    "weight_decay": REAL_WEIGHT_DECAY,
    "adam_betas": REAL_ADAM_BETAS,
    "adam_epsilon": REAL_ADAM_EPSILON,
    "max_grad_norm": REAL_MAX_GRAD_NORM,
    "warmup_steps": REAL_WARMUP_STEPS,
    "eval_and_save_interval": REAL_EVAL_INTERVAL,
    "precision": "FP32（fp16=False, bf16=False, tf32=False）",
    "device": str(real_training_arguments.device),
    "artifact_root": str(real_artifact_root),
    "runtime_target": "约 1～3 分钟（常见 Colab/Apple 加速器；CPU 通常更久）",
}

real_pipeline_contract = {
    "filter": real_asset_summary["segment_filter"],
    "document_boundary": "append exactly one GPT-2 EOS to every nonempty segment",
    "packing": "ordered carry buffer; fixed 128 tokens; drop only final split tail",
    "validation_selection": f"shuffle(seed={SEED}) then first {REAL_EVAL_BLOCK_LIMIT} blocks",
    "test_policy": real_asset_summary["test_policy"],
}
real_pipeline_sha256 = hashlib.sha256(
    json.dumps(real_pipeline_contract, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()

real_output_dir.mkdir(parents=True, exist_ok=False)
real_run_manifest = {
    "assets": real_asset_summary,
    "packing": real_packing_summary,
    "budget": real_run_budget,
    "pipeline_contract": real_pipeline_contract,
    "pipeline_contract_sha256": real_pipeline_sha256,
    "model_config": real_model_config.to_dict(),
    "training_arguments": real_training_arguments.to_dict(),
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "datasets": datasets.__version__,
        "transformers": transformers.__version__,
    },
}
(real_output_dir / "run_manifest.json").write_text(
    json.dumps(real_run_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

real_trainer = Trainer(
    model=real_model,
    args=real_training_arguments,
    train_dataset=real_packed_corpus["train"],
    eval_dataset=real_packed_corpus["validation"],
    data_collator=default_data_collator,
    processing_class=real_tokenizer,
)
real_run_budget

### 5.5．执行训练并比较固定留出验证指标

训练前先记录随机初始化基线，训练中保留 500-step 间隔的验证记录，训练后对自动恢复的最佳 Checkpoint 再评估。同一 validation 子集同时参与最佳 Checkpoint 选择，因此它是与训练数据分离的固定留出集，却不是独立于模型选择的最终测试集；test 仍保持封存。

`TrainOutput.training_loss` 是训练期间均值，不是质量验收；`samples/s` 的 sample 是 packed block，也不能跨 block size 直接比较。本单元额外报告有效目标 token/s。默认会实际执行 1,500 次更新，并写入环境变量 `LLM_FROM_SCRATCH_ARTIFACT_ROOT` 指定的目录；未设置时使用 `artifacts/40_pre_training/gpt2-wikitext2-small/<UTC run id>`。

默认路径是一次全新训练，不接受任意 Checkpoint 的隐式拼接。生产恢复需要读取既有 run 目录，逐项校验源文件 SHA-256、pipeline contract SHA-256、模型配置和完整 `TrainingArguments`，然后以同一输出目录调用 `real_trainer.train(resume_from_checkpoint=...)`；第 4 节给出了完整状态与 RNG 的恢复语义。

In [ ]:
import time

baseline_metrics = real_trainer.evaluate(metric_key_prefix="baseline")
wall_start = time.perf_counter()
real_train_output = real_trainer.train()
training_wall_seconds = time.perf_counter() - wall_start
final_metrics = real_trainer.evaluate(metric_key_prefix="final")

if real_train_output.global_step != REAL_MAX_STEPS:
    raise RuntimeError(
        f"训练应完成 {REAL_MAX_STEPS:,} 次更新，实际为 {real_train_output.global_step:,}。"
    )
if not math.isfinite(baseline_metrics["baseline_loss"]):
    raise RuntimeError("随机初始化基线损失不是有限值。")
if not math.isfinite(final_metrics["final_loss"]):
    raise RuntimeError("训练后验证损失不是有限值。")
if final_metrics["final_loss"] >= baseline_metrics["baseline_loss"]:
    raise RuntimeError("验证损失未低于随机初始化基线；请检查数据、学习率和训练日志。")

step_evaluations = [
    {"step": int(record["step"]), "eval_loss": record["eval_loss"]}
    for record in real_trainer.state.log_history
    if "eval_loss" in record
]
train_runtime = real_train_output.metrics["train_runtime"]
real_training_evidence = {
    "global_step": real_train_output.global_step,
    "training_loss": real_train_output.training_loss,
    "baseline_eval_loss": baseline_metrics["baseline_loss"],
    "final_best_checkpoint_eval_loss": final_metrics["final_loss"],
    "baseline_perplexity": math.exp(baseline_metrics["baseline_loss"]),
    "final_perplexity": math.exp(final_metrics["final_loss"]),
    "effective_target_tokens": real_target_token_budget,
    "effective_target_tokens_per_second": real_target_token_budget / train_runtime,
    "trainer_runtime_seconds": train_runtime,
    "training_wall_seconds_including_eval_and_save": training_wall_seconds,
    "best_checkpoint": real_trainer.state.best_model_checkpoint,
    "step_evaluations": step_evaluations,
    "artifact_dir": str(real_output_dir),
}
real_training_evidence

### 5.6．机制可视化：调度轨迹与训练证据

**学习问题**：Warmup—Cosine 调度在真实更新步上如何变化，训练日志与固定验证集证据是否呈现一致的学习趋势？

**验收不变量**：图中训练损失、验证损失和学习率全部直接读取本次 `real_trainer.state.log_history`；横轴统一使用 `Trainer` 记录的 `step`，验证点只来自训练期间按 `REAL_EVAL_INTERVAL` 触发的同一冻结 validation 子集。可视化不插值、不补写目标曲线，也不以最终最佳 Checkpoint 的复评值替代训练期间记录。


In [ ]:
training_log_points = [
    record for record in real_trainer.state.log_history
    if "step" in record and "loss" in record and "learning_rate" in record
]
validation_log_points = [
    record for record in real_trainer.state.log_history
    if "step" in record and "eval_loss" in record
]
if not training_log_points or not validation_log_points:
    raise RuntimeError("Trainer log_history 缺少训练损失、学习率或验证损失记录")
if any(int(record["step"]) > REAL_MAX_STEPS for record in training_log_points + validation_log_points):
    raise RuntimeError("日志包含超出预注册训练预算的更新步")

train_steps = [int(record["step"]) for record in training_log_points]
train_losses = [float(record["loss"]) for record in training_log_points]
logged_learning_rates = [float(record["learning_rate"]) for record in training_log_points]
validation_steps = [int(record["step"]) for record in validation_log_points]
validation_losses = [float(record["eval_loss"]) for record in validation_log_points]

figure, (loss_axis, lr_axis) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
loss_axis.plot(train_steps, train_losses, color="#2563eb", linewidth=1.5, label="训练日志 loss")
loss_axis.scatter(validation_steps, validation_losses, color="#dc2626", s=48, zorder=3, label="固定验证集 loss")
loss_axis.set(ylabel="cross-entropy", title="真实训练与验证损失记录")
loss_axis.grid(alpha=0.25)
loss_axis.legend()

lr_axis.plot(train_steps, logged_learning_rates, color="#059669", linewidth=1.8, label="Trainer 记录的 learning rate")
lr_axis.axvline(REAL_WARMUP_STEPS, color="#64748b", linestyle="--", linewidth=1.0, label="Warmup 边界")
lr_axis.set(xlabel="optimizer step", ylabel="learning rate", title="Warmup—Cosine 实际日志轨迹")
lr_axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
lr_axis.grid(alpha=0.25)
lr_axis.legend()
plt.show()


**应观察结论**：学习率在前 `REAL_WARMUP_STEPS` 个更新步上升，随后按 Cosine 调度下降；训练日志损失提供优化过程证据，间隔验证点提供未参与梯度更新的固定留出证据。满足本章验收时，训练后的验证损失低于随机初始化基线，且日志步数不超过 `REAL_MAX_STEPS`。

**不可误读边界**：训练 `loss` 是相邻日志窗口内的批次统计，验证 `eval_loss` 是完整固定子集的聚合，两条曲线采样频率与统计口径不同，不能逐点相减。曲线下降只验证当前数据—目标—优化链路正在学习；本 validation 同时用于选择最佳 Checkpoint，不能替代封存测试集、下游能力评测或多 Seed 质量结论。


## 6．生产边界

### 6.1．本章真实训练仍不是可用基座模型

第 5 节已经使用真实公开语料、正式 Tokenizer、标准 Causal LM、验证集和可恢复 Checkpoint，但仍是受控的小型实验：约 724 万参数只展示约 152 万个目标 token，Token/参数比、语料覆盖、上下文长度与评估广度都不足以产出可用语言模型。验证损失下降只能证明当前数据—模型—优化链路在学习，不能证明事实性、中文能力、安全性或下游泛化。

WikiText-2 来自 Wikipedia 优良 / 特色文章，适合小型可审计语言建模实验，但含旧式预处理与 WikiText 标记，不代表 2026 年生产语料配方。真实交付还必须完成许可证与来源治理、去重和污染检查、PII / 安全过滤、质量分层、数据 Manifest、多个 seed、封存测试集与下游评测。`samples/s` 以 packed block 为单位；生产吞吐应同时报告有效目标 token/s、global batch、world size、精度、设备、峰值显存和利用率。每个 run 仅在自身目录内保留两个 Checkpoint，跨 run 的保留、归档与删除策略仍需由制品生命周期策略管理；同步盘写入会改变墙钟时间。

### 6.2．从单进程闭环扩展到分布式训练

分布式训练不是另一套目标函数，而是把参数、梯度、Optimizer State、Activation、序列或专家路由分配到多个设备。选择策略前先判断**哪一类状态放不下或哪一段链路吞吐不足**。

```mermaid
flowchart TD
    A["单进程语义基线"] --> B{"主要瓶颈"}
    B -->|"数据吞吐 / 算力"| C["数据并行 DDP"]
    B -->|"Optimizer State / 梯度 / 参数"| D["ZeRO / FSDP"]
    B -->|"单层参数过大"| E["Tensor Parallel"]
    B -->|"层数与 Activation"| F["Pipeline Parallel"]
    B -->|"超长上下文"| G["Context / Sequence Parallel"]
    B -->|"稀疏专家容量"| H["Expert Parallel / MoE"]
    C --> I["混合并行"]
    D --> I
    E --> I
    F --> I
    G --> I
    H --> I
```

无论采用哪种并行，Global Batch、有效 token 数、学习率曲线、梯度同步边界和 Checkpoint 一致性都必须可解释。DDP、DeepSpeed ZeRO-1/2/3、FSDP 与混合并行的实现和取舍进入 [A40_training_optimization.ipynb](A40_training_optimization.ipynb)。

### 6.3．训练主线的五个不变量

1. **数据不变量**：数据版本、文件哈希、split、过滤规则和 token 顺序可追溯。
2. **输入不变量**：Tokenizer、特殊 token、上下文长度与文档边界形成统一协议。
3. **目标不变量**：Decoder-only 预训练始终用过去 token 预测下一个 token，labels 在模型内部移位。
4. **更新不变量**：Global Batch、有效 token 数、Optimizer Step 与学习率调度使用同一计量口径。
5. **恢复不变量**：Checkpoint 恢复完整训练轨迹；仅有权重只能开始新的训练阶段，不能声称精确续训。

继续学习时按问题分流：

- 要把基座模型变成会遵循指令并完成偏好对齐的任务模型：进入 [41_post_training.ipynb](41_post_training.ipynb)；
- 要降低显存、提高吞吐或扩展到多机多卡：进入紧随本章的 [A40_training_optimization.ipynb](A40_training_optimization.ipynb)。

### 6.4．参考资料

- [WikiText 数据集卡](https://huggingface.co/datasets/Salesforce/wikitext)
- [WikiText-2 数据集](https://huggingface.co/datasets/Salesforce/wikitext)
- [GPT-2 Tokenizer](https://huggingface.co/openai-community/gpt2)
- [Hugging Face Datasets：Dataset 与 DatasetDict](https://huggingface.co/docs/datasets/package_reference/main_classes)
- [Hugging Face Transformers：Trainer](https://huggingface.co/docs/transformers/main_classes/trainer)
- [PyTorch：torch.accelerator](https://docs.pytorch.org/docs/stable/accelerator.html)